# DAC/ADC loopback: direct RF and D2S + mixer
One A8 bitstream, **DAC 229/1 → ADC 224/3 in both modes**. The external mixer LO remains TP DAC 228/1 (PYNQ tile 0, block 1, `axi_gpio_12`). The intensity ADC is not used. Run cells individually with laser controllers disconnected from the driven DAC. Use your usual board-clock initialization. Loading the overlay interrupts existing feedback.

Start with the same **90 MHz carrier** in both modes to compare their delay. Direct RF may additionally be tested at 800 MHz. The ZAD-1+ cannot be used at 800 MHz. Its specified LO is +7 dBm; verify power at its LO connector rather than inferring it from `lo_amp`.

- `rf`: DAC 229/1 RF output → suitable RF connection/attenuation → ADC 224/3.
- `mixer`: DAC 229/1 near DC → D2S board → optional BLP → ZAD-1+ IF; TP LO → LO; ZAD-1+ RF → ADC 224/3.

Keep BLP/cables/terminations recorded. The mixer IF supplies a nominal 50 Ω load. With the mixer removed, the high-impedance FALC/scope/PI tee does not supply that load.


In [ ]:
from pathlib import Path
from datetime import datetime
import importlib
import json
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay
import xrfdc
import laser_pll
import laser_pll_loop_sweep as fra
import laser_pll_loopback as lb
laser_pll = importlib.reload(laser_pll)
fra = importlib.reload(fra)
lb = importlib.reload(lb)
fra.laser_pll = laser_pll
lb.laser_pll = laser_pll
lb.fra = fra
# Perform your normal RFSoC reference-clock initialization before loading.
ol = Overlay('laser_pll.bit')
print('Helper:', laser_pll.__file__)
print('FPGA version:', hex(ol.pll_acquisition_status.read(8) >> 24))
assert laser_pll.FPGA_VERSION == 0xA8
assert ol.pll_acquisition_status.read(8) >> 24 == 0xA8


## Select the physical test
After moving cables, rerun setup and calibration. `setup()` leaves the RFDC ADC rate/mixer/decimation and TP LO unchanged; it sets the fabric receiver reference and the measured DAC NCO. The primary PLL becomes a stimulus source in DC-test mode. The second PLL is not reconfigured.


In [ ]:
mode = 'mixer'                 # 'rf' for direct DAC → ADC
carrier_hz = 90e6              # also use 90 MHz for the direct comparison
bias_codes = 2048              # nominal +62.5 mV for a 2 Vpp DAC
amplitude_codes = 256          # nominal 7.8125 mV peak; start small
if mode == 'mixer':
    # Your existing TP LO; verify approximately +7 dBm at the ZAD-1+ LO port.
    lo_freq = carrier_hz / 1e6  # RFDC mixer frequency is in MHz
    lo_amp = 0.25
    dac_block = ol.usp_rf_data_converter_0.dac_tiles[0].blocks[1]
    mixer_cfg = dict(dac_block.MixerSettings)
    mixer_cfg['Freq'] = lo_freq
    dac_block.MixerSettings = mixer_cfg
    dac_block.UpdateEvent(xrfdc.EVENT_MIXER)
    ol.axi_gpio_12.channel1.write(int(lo_amp * 32767), mask=0xFFFF)
setup_info = lb.setup(ol, mode=mode, carrier_hz=carrier_hz, bias_codes=bias_codes)
print(lb.receiver_settings(ol))


## DC calibration and first point
Calibration measures two nearby DAC bias levels in a low–high–low sequence. Subtraction removes fixed LO leakage and receiver DC offsets. It records the real I/Q response vector and rejects excessive drift. The result is normalized to that DC slope, including polarity.

If calibration drifts or vanishes, inspect LO level, cabling, DC coupling and matching carrier frequencies before running a long sweep. This is an amplitude-response measurement; it does not use CORDIC or the PLL discriminator.


In [ ]:
calibration = lb.calibrate(ol, step_codes=512, integration_s=1.0)
print({k: calibration[k] for k in ('gain_i', 'gain_q', 'drift_fraction')})
point = lb.measure_point(ol, 1e3, amplitude_codes=amplitude_codes, minimum_measure_s=1.0)
h, perpendicular = lb.project(point, calibration)
print('Actual Hz:', point['frequency_hz'], 'H:', h, 'orthogonal response:', perpendicular)
print('Diagnostics:', point['diagnostics'])


## Long coherent sweep
Mixing with the reference happens at 245.76 MHz **before** integrating to one result per block. Default one-second blocks give approximately 0.5 Hz one-sided rectangular-window equivalent noise bandwidth per demodulated quadrature. Eight independent blocks reduce uncorrelated repeat noise by another √8; correlated drift and distortion do not average away.

The default 61-point sweep takes roughly 9–27 minutes depending on adaptive repeats; low-frequency whole-cycle integration can take longer. It saves CSVs after every block/point in the notebook's `loopback_data` folder. No raw GHz time series is transferred over DMA. Flags identify unreliable points; no smoothing conceals them.


In [ ]:
output_dir = Path('loopback_data')
output_dir.mkdir(exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = output_dir / f'loopback_{mode}_{stamp}.csv'
metadata = dict(mode=mode, carrier_hz=carrier_hz, bias_codes=bias_codes,
                fpga_version=hex(ol.pll_acquisition_status.read(8) >> 24),
                fabric_sample_rate_hz=lb.FS_HZ, iq_units='post-mixer ADC counts',
                amplitude_codes=amplitude_codes, gain_i=calibration['gain_i'],
                gain_q=calibration['gain_q'], drift_fraction=calibration['drift_fraction'],
                settings=lb.receiver_settings(ol),
                fixture_notes='EDIT: D2S, BLP present/absent, cables, attenuation, mixer LO power')
csv_path.with_suffix('.json').write_text(json.dumps(metadata, indent=2))
frequencies_hz = np.geomspace(10, 1e6, 61)
rows = lb.sweep(ol, frequencies_hz, calibration,
                amplitude_codes=amplitude_codes, minimum_measure_s=1.0,
                repeats=8, max_repeats=24, target_snr=30, csv_path=csv_path)
print('Saved:', csv_path.resolve())
print('Raw complex blocks:', csv_path.with_name(csv_path.stem + '_blocks.csv').resolve())


In [ ]:
frequency_hz = np.array([r['frequency_hz'] for r in rows])
h = np.array([complex(r['h_real'], r['h_imag']) for r in rows])
sem = np.array([r['complex_sem'] for r in rows])
reliable = np.array([r['reliable'] for r in rows])
phase_deg = np.rad2deg(np.unwrap(np.angle(h)))
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
axes[0].errorbar(frequency_hz, 20*np.log10(abs(h)),
                 yerr=20/np.log(10)*sem/abs(h), fmt='.-', label=mode)
axes[1].errorbar(frequency_hz, phase_deg, yerr=np.rad2deg(sem/abs(h)), fmt='.-')
axes[0].plot(frequency_hz[~reliable], 20*np.log10(abs(h[~reliable])), 'rx', label='low SNR')
axes[0].set_ylabel('DC-normalized magnitude (dB)')
axes[1].set_ylabel('Unwrapped phase (deg)')
axes[1].set_xlabel('Modulation frequency (Hz)')
for ax in axes:
    ax.set_xscale('log'); ax.grid(True, which='both', alpha=.25)
y_low, y_high = axes[0].get_ylim()
if y_high-y_low < 1.0:
    middle = (y_low+y_high)/2
    axes[0].set_ylim(middle-.5, middle+.5)
axes[0].legend()
fig.suptitle(f'{mode} loopback, {carrier_hz/1e6:g} MHz carrier')
fig.tight_layout()
fig.savefig(csv_path.with_suffix('.png'), dpi=180)
delay = lb.fit_delay(rows, minimum_hz=1e4, maximum_hz=1e6)
print(f"Fitted total delay: {delay['delay_s']*1e9:.1f} ns")
print(f"Phase-line residual RMS: {np.rad2deg(delay['phase_residual_rms_rad']):.2f} deg")


## Checks and interpretation
1. Repeat selected points at half amplitude. Their **normalized complex responses** should agree within the uncertainty; otherwise suspect compression, rail events or poor SNR.
2. Repeat the DC calibration after the sweep. A changed I/Q direction indicates carrier phase drift; a changed magnitude indicates gain drift. Keep converter/NCO state fixed within a sweep.
3. Measure both modes at the same 90 MHz carrier. Divide the mixer-path complex response by the direct response, then unwrap phase. This removes much of the shared digital/converter contribution; different external RF cables, the mixer, and DAC analog response near DC versus RF still contribute.
4. An extra known cable should add its propagation delay (roughly 5 ns/m for many coax types; use that cable's velocity factor). Repeat after RFDC resets to check startup-dependent latency.
5. For a dispersive LPF, the phase curve bends: a single delay fit is only a band-average slope. Inspect the curve and residual; do not equate group delay at one frequency with phase delay everywhere.

The measurement includes DAC packing/interpolation, DAC/ADC conversion and the existing mixer/FIR. It excludes CORDIC, phase unwrapping, phase scaling, analog controllers and laser dynamics. No software-trigger timing enters the phase estimate. Absolute delay cannot distinguish DAC from ADC latency.

Run this cleanup cell before changing wiring. It mutes the measured DAC; it leaves the TP LO and second PLL as configured.


In [ ]:
lb.stop(ol, mute=True)
# To resume a laser lock, run the ordinary PLL configure/acquisition cells.
# laser_pll.configure() restores the measured DAC NCO to zero frequency/phase.
